# Agent-based swarm models: student quickstart

Use this notebook when the agents and their interaction rules are the object of
study. It shows the complete ABM workflow:

1. choose a model and operating regime;
2. simulate one reproducible trajectory;
3. inspect motion and collective-order statistics;
4. compare regimes or sweep parameters without overwriting saved validation.

Related notebooks:

- `Tutorial_ESN.ipynb`: conventional echo-state networks;
- `Tutorial_swarmRC.ipynb`: use a swarm as a physical reservoir;
- `advanced/Existing_Model_Validation.ipynb`: validation against published models.

In [ ]:
for d in ("FIGURES/ABM", "ANIMATIONS/ABM")
    mkpath(d)
end

## 1. Choose a model

| Model | State and domain | Interactions | Use it when… |
|---|---|---|---|
| **Couzin** | positions and headings; 2D periodic adaptation or 3D model | repulsion, orientation and attraction zones | collective-motion regimes and interaction networks |
| **Lymburn** | 2D positions and velocities; home-centred | repulsion, alignment, homing, friction and optional predator | force-driven flocking and swarm reservoirs |
| **Mizzi** | 2D positions and velocities; fixed Voronoi territories | homing, friction and territory-gated prey attraction | distinguishable territorial agents |
| **Helbing** | 2D or 3D pedestrian positions and velocities | desired velocity, social/contact forces and boundaries | pedestrian flow and congestion |

All models use the same simulation interface and return positions, velocities, time and order parameters. Presets are model-specific; an incompatible preset is replaced by that model's default with a printed notice.

**Mizzi:** the example constructs homes from a simple embedded prey trajectory. Supply a study-specific trajectory for research use.

**Lymburn:** the Euler step is fixed at $\Delta t=0.02$. Swarm size affects whether organised motion emerges; use $N\geq100$ when inspecting the `:critical` regime.

In [ ]:
abm_choice = (
    model = :Couzin,
    dimension = 2,
    preset = :milling,
    N = 50,
    L = 50.0,
    dt = 0.1,
    steps = 1000,
    seed = 1,
    animation_stride = 5,
    overwrite_animation = false,
)

## 2. Construct and simulate the selected system

Model loaders expose model-specific parameter constructors and simulators.
The generic experiment layer provides a shared interface for controlled
parameter sweeps.

In [ ]:
include("ABM/load_ABM.jl")
include("ABM/models/Couzin/load_Couzin.jl")
include("ABM/models/Lymburn/load_Lymburn.jl")
include("ABM/models/Helbing/load_Helbing.jl")
include("ABM/models/Mizzi/load_Mizzi.jl")
include("ABM/my_ABM_experiments.jl")

In [ ]:
"""
    _resolve_preset(valid_presets, model, requested, fallback) -> Symbol

`preset` (like `model`) is model-specific vocabulary -- Couzin's
`:milling` means nothing to Lymburn, and vice versa. Switching `model`
alone, with `preset` left at another model's value, is a very natural
thing to try and previously just crashed with a bare "unknown preset"
error. Fall back to a sensible default instead, and say so, rather than
silently ignoring the mismatch or erroring uninformatively.
"""
function _resolve_preset(valid_presets, model, requested, fallback)
    if requested in valid_presets
        return requested
    end
    valid_str = join(sort(String.(collect(valid_presets))), ", ")
    println("Note: :$requested is not a $model preset (valid: $valid_str); using :$fallback instead.")
    return fallback
end

"""
    run_abm(choice) -> (out, axis_limits, periodic)

Dispatch to the right per-model parameter constructor and simulator. Every
model funnels through the same generic `simulate` driver, so `out` always
has `pos_hist`, `vel_hist`, `t`, `dilation`, `rotation`, `polarisation` and
`speed`, regardless of which model produced it. `axis_limits` is
`(xlo, xhi, ylo, yhi)`, for display only -- Couzin/Helbing use a `[0, L]`
periodic-style box, but Lymburn/Mizzi's `plot_extent` is a *half*-range
around a home position (default the origin), so the two conventions are
not interchangeable and each branch returns limits in its own convention
rather than a single bare number. `periodic` controls whether trails wrap.
"""
function run_abm(choice)
    if choice.model == :Couzin
        preset = _resolve_preset(keys(COUZIN_REGIME_PRESETS), :Couzin, choice.preset, :milling)
        scenario = Couzin_params_from_preset(
            preset;
            dim=choice.dimension, N=choice.N, L=choice.L, dt=choice.dt,
        )
        simcfg = SimulationConfig(steps=choice.steps, dt=scenario.dt, seed=choice.seed)
        out = choice.dimension == 2 ?
            simulate_Couzin_2d(simcfg, scenario.P; show_progress=false) :
            simulate_Couzin_3d(simcfg, scenario.P; show_progress=false)
        L = scenario.P.L
        return out, (0.0, L, 0.0, L), true

    elseif choice.model == :Lymburn
        preset = _resolve_preset(keys(LYMBURN_PRESETS), :Lymburn, choice.preset, :critical)
        P = Lymburn_params_from_preset(preset; N=choice.N)

        # Lymburn's force model uses (semi-implicit) Euler integration,
        # which needs a much finer step than Couzin's turning-rate update
        # tolerates -- checked directly: at the switchboard's shared
        # dt=0.1, the paper's own "critical" (milling-like) regime showed
        # Phi_R = 0.02-0.09 (no organisation at all, regardless of N); at
        # the paper's own dt=0.02, Phi_R = 0.98-1.0 (fully organised).
        # This is a numerical-stability requirement of the model, not a
        # scientific choice, so it overrides choice.dt below rather than
        # respecting it -- the same pattern Couzin/Helbing already use via
        # scenario.dt.
        lymburn_dt = 0.02

        # N is a genuine scientific choice, unlike dt, so it is not
        # overridden -- but the effect is not subtle and worth flagging:
        # checked directly (dt=0.02, everything else fixed), N=50 gives
        # Phi_R = 0.0, N=100 gives Phi_R = 1.0. The switchboard's shared
        # default N=50 (fine for Couzin milling) will show no organisation
        # at all for Lymburn's :critical regime.
        if preset == :critical && choice.N < 100
            println("Note: Lymburn's :critical (milling-like) regime was validated at N=200 " *
                     "and needs roughly N>=100 to organise at all. N=$(choice.N) will very " *
                     "likely show no organisation (checked directly: N=50 gives rotation~0, " *
                     "N=100 gives rotation~1.0, with dt and everything else held fixed).")
        end

        simcfg = SimulationConfig(steps=choice.steps, dt=lymburn_dt, seed=choice.seed)
        out = simulate_Lymburn_2d(simcfg, P; show_progress=false)
        e = P.plot_extent
        return out, (P.xh.x - e, P.xh.x + e, P.xh.y - e, P.xh.y + e), false

    elseif choice.model == :Helbing
        preset = _resolve_preset(keys(HELBING_REGIME_PRESETS), :Helbing, choice.preset, :lane_formation)
        scenario = Helbing_params_from_preset(preset; N=choice.N, dim=choice.dimension)
        simcfg = SimulationConfig(steps=choice.steps, dt=scenario.dt, seed=choice.seed)
        out = choice.dimension == 2 ?
            simulate_Helbing_2d(simcfg, scenario; show_progress=false) :
            simulate_Helbing_3d(simcfg, scenario; show_progress=false)
        L = scenario.P.L
        return out, (0.0, L, 0.0, L), false

    elseif choice.model == :Mizzi
        # Mizzi's territorial homes are sampled from an embedded prey
        # trajectory, not from a named preset -- the one genuine structural
        # difference among these four models (a required extra input, not
        # just a different parameter shape). A simple synthetic circular
        # prey path keeps this switchboard fully generic for a first look;
        # supply your own real trajectory for anything beyond that (Section 5).
        T = choice.steps + 1
        θ = range(0, 4π; length=T)
        r = choice.N / 2
        U = permutedims(hcat(r .* cos.(θ), r .* sin.(θ)))
        homes = mizzi_homes_from_input(U, choice.N; rng=MersenneTwister(choice.seed))
        P = MizziParams(homes)
        simcfg = SimulationConfig(steps=choice.steps, dt=choice.dt, seed=choice.seed)
        out = simulate_Mizzi_2d(simcfg, P; prey=U, show_progress=false)
        e = P.plot_extent
        return out, (-e, e, -e, e), false

    else
        error("Unknown model $(choice.model). Supported: :Couzin, :Lymburn, :Helbing, :Mizzi.")
    end
end

In [ ]:
out_abm, axis_limits_abm, periodic_abm = run_abm(abm_choice)

(frames=length(out_abm.pos_hist), agents=length(out_abm.pos_hist[1]),
 final_time=out_abm.t[end])

## 3. Inspect the trajectory

A trajectory is not characterised by its final frame alone. Use snapshots,
animation and order parameters together: snapshots show geometry, animation
shows whether that geometry persists, and order parameters make comparisons
quantitative.

In [ ]:
snapshot_times = [0.0, 0.25 * out_abm.t[end], 0.5 * out_abm.t[end], out_abm.t[end]]
snapshot_frames = [argmin(abs.(out_abm.t .- t)) for t in snapshot_times]
xlo, xhi, ylo, yhi = axis_limits_abm
periodic_L_abm = periodic_abm ? (xhi - xlo) : nothing

if abm_choice.dimension == 2
    fig_abm = Figure(size=(1100, 290))
    for (j, k) in enumerate(snapshot_frames)
        ax = Axis(fig_abm[1, j], title="t=$(round(out_abm.t[k], digits=1))", aspect=DataAspect())
        plot_swarm_frame!(ax, out_abm.pos_hist, out_abm.vel_hist, k;
            trail_len=12, agent_ms=6.0, periodic_L=periodic_L_abm)
        xlims!(ax, xlo, xhi); ylims!(ax, ylo, yhi)
    end
    fig_abm
else
    println("2D snapshot panels only; for 3D Couzin see advanced/Existing_Model_Validation.ipynb's B2 section.")
end

In [ ]:
abm_animation_path = "ANIMATIONS/ABM/$(abm_choice.model)_$(abm_choice.preset)_$(abm_choice.dimension)d.mp4"
animcfg_abm = AnimationConfig(
    fps=30, stride=abm_choice.animation_stride, shaft_len=0.45, agent_ms=7.0,
    save_mp4=true, filename=abm_animation_path,
    fig_size=(1100, 650), show_zones=false,
)
if abm_choice.overwrite_animation || !isfile(abm_animation_path)
    title_abm = "$(abm_choice.model) $(abm_choice.preset) regime"
    abm_choice.dimension == 2 ?
        animate_simulation_2d(out_abm, animcfg_abm; title=title_abm) :
        animate_simulation_3d(out_abm, animcfg_abm; title=title_abm)
else
    println("Using saved animation: $abm_animation_path")
end
abm_animation_path

In [ ]:
display(plot_order_parameters(out_abm))

order_summary = mean_order_parameters(out_abm; transient_steps=200)
println("mean rotation:     ", round(order_summary.rotation_mean, digits=3))
println("mean polarisation: ", round(order_summary.polarisation_mean, digits=3))
println("mean dilation:     ", round(order_summary.dilation_mean, digits=3))

For a milling group, sustained rotation should be high while global
polarisation remains comparatively low. If a visually attractive trajectory
does not produce the expected quantitative signature—or changes completely
with the seed—do not treat the preset label as evidence.

## 4. Compare regimes and parameter space

One hand-picked trajectory is not evidence of a regime; a parameter map is.
This section is Couzin/Couzin(2002)-specific by nature (it reproduces a
specific published phase diagram), so the actual sweep code lives in
`advanced/Existing_Model_Validation.ipynb` and is only *displayed* here, as the
protected, full validation result:

![Couzin 2D regimes](FIGURES/ABM/couzin2002_2d_regimes.png)

![Couzin 2D parameter map](FIGURES/ABM/couzin2002_2d_phase_full_heatmap.png)

To run a sweep yourself -- for Couzin or any other model -- use
`advanced/Existing_Model_Validation.ipynb`, which already has the full
`run_model_experiment`/`summarise_experiment` machinery at both a fast
exploratory scale and the paper-faithful scale. There is no separate
lightweight copy of that machinery here to keep in sync.</cell id="abm-parameter-heading">


## 5. Beyond the switchboard

`abm_choice.model` (Section 1) already covers the common case for all four
models directly. This section is for going further than the switchboard's
defaults can:

```julia
# Mizzi: territorial homes from your own embedded trajectory, not the
# switchboard's synthetic circular default
homes = mizzi_homes_from_input(U, 30; rng=MersenneTwister(1))   # U: your own 2×T prey trajectory
P = MizziParams(homes)
out = simulate_Mizzi_2d(SimulationConfig(steps=1000, dt=0.02, seed=1), P; prey=U)
```

Predator-driven Lymburn (`Kp != 0`) is not a plain-`simulate_Lymburn_2d`
option -- the predator is wired through the swarm-reservoir driving
machinery, not a runtime trajectory argument here. See
`ABM/models/Lymburn/my_Lymburn_experiments.jl` and `Tutorial_swarmRC.ipynb`
for that path.

Before extending a project, record the model, dimension, boundary condition,
preset, timestep, seed, transient removal and ensemble size. Those choices are
part of the result.</cell id="abm-alternatives">


## Where to go next

- Use `advanced/Existing_Model_Validation.ipynb` for paper-faithful Couzin results.
- Use `Tutorial_TDA.ipynb` for topological observations of swarm states.
- Continue to `Tutorial_swarmRC.ipynb` when the swarm becomes a driven
  information-processing substrate rather than only the object being modelled.